# Techniques SSL avancées - FixMatch, FlexMatch et MixMatch

***Principes clés :***

- FixMatch : Utilise des augmentations faibles et fortes avec un seuil de confiance pour les pseudo‑labels.
- FlexMatch : Améliore FixMatch avec un seuillage dynamique par classe, idéal pour les données déséquilibrées.
- MixMatch : Ajoute du mélange de données (ex. MixUp) pour améliorer la robustesse en combinant échantillons étiquetés et non étiquetés.
Objectifs :

Revenir à la classification DermaMNIST avec 100 images étiquetées.
Implémenter FixMatch, FlexMatch et MixMatch.
Comparer les résultats au baseline afin d’illustrer les avancées SSL.

## 1. Préparation des données

Mettons en place l’environnement pour la classification DermaMNIST. Nous utiliserons 100 images étiquetées et Albumentations pour des augmentations contrôlées.

In [2]:
# importer les packages
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
!pip install medmnist
!pip install albumentations
import medmnist
from medmnist import INFO, Evaluator
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.4/110.4 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.4/369.4 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 844.5/844.5 kB 39.4 MB/s eta 0:00:00


In [3]:
# Load DermaMNIST data
data_flag = 'dermamnist'
info = INFO[data_flag]
n_classes = len(info['label'])
DataClass = getattr(medmnist, info['python_class'])

train_dataset = DataClass(split='train', download=True)
test_dataset = DataClass(split='test', transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=[.5], std=[.5])]), download=True)

# Split into labeled (100) and unlabeled sets
all_indices = list(range(len(train_dataset)))
labels_array = np.array(train_dataset.labels).flatten()
labeled_indices, unlabeled_indices = train_test_split(all_indices, train_size=500, random_state=42, stratify=labels_array)

print(f"Données étiquetées : {len(labeled_indices)}, Données non étiquetées : {len(unlabeled_indices)}")

100%|██████████| 19.7M/19.7M [00:28<00:00, 683kB/s]


Données étiquetées : 500, Données non étiquetées : 6507


## Augmentations faibles et fortes

Nous avons besoin de deux pipelines d’augmentation : faible pour la génération de pseudo‑labels et forte pour accroître la robustesse à l’entraînement.

In [4]:
# Define weak and strong augmentations for single-channel images
weak_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ToTensorV2(transpose_mask=True)  # Preserve 1 channel
])

strong_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussianBlur(p=0.3),
    A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ToTensorV2(transpose_mask=True)  # Preserve 1 channel
])
print("Transformations initialisées")

# Custom datasets for FixMatch
class FixMatchDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = Subset(dataset, indices)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img = np.array(img)  # Ensure img is [H, W] (single-channel)
        transformed = self.transform(image=img)
        return transformed['image'], torch.tensor(label).long()

class FixMatchUnlabeledDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, weak_transform, strong_transform):
        self.dataset = Subset(dataset, indices)
        self.weak_transform = weak_transform
        self.strong_transform = strong_transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        img = np.array(img)  # Ensure img is [H, W] (single-channel)
        weak = self.weak_transform(image=img)['image']
        strong = self.strong_transform(image=img)['image']
        return weak, strong

Transformations initialisées


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


## 2. Modèles et boucles d’entraînement

Nous utiliserons un CNN simple et implémenterons trois boucles d’entraînement : FixMatch, FlexMatch et MixMatch.

In [5]:
# Define the SimpleCNN model for single-channel input
class SimpleCNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(SimpleCNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Linear(7 * 7 * 32, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        return self.fc(out)

# Initialize model, optimizer, and loss functions
model = SimpleCNN(in_channels=3, num_classes=n_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

supervised_criterion = nn.CrossEntropyLoss()
unsupervised_criterion = nn.CrossEntropyLoss(reduction='none')

## ⚙️ 2.1 Boucle d’entraînement FixMatch
Implémentons l’algorithme FixMatch pas à pas.

Instructions :

- Calculer la perte supervisée sur les données étiquetées.
- Générer des pseudo‑labels : prédire sur les augmentations faibles, calculer les probabilités et créer un masque pour les prédictions confiantes (seuil = 0.95).
- Calculer la perte non supervisée : prédire sur les augmentations fortes et appliquer le masque aux pseudo‑labels confiants.
- Combiner les pertes et faire la rétropropagation.

In [6]:
# create Dataloader
labeled_dataset = FixMatchDataset(train_dataset, labeled_indices, strong_transform)
unlabeled_dataset = FixMatchUnlabeledDataset(train_dataset, unlabeled_indices, weak_transform, strong_transform)

print(f"Taille du dataset étiqueté : {len(labeled_dataset)}")
print(f"Taille du dataset non étiqueté : {len(unlabeled_dataset)}")

labeled_loader = DataLoader(labeled_dataset, batch_size=16, shuffle=True)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True)
print(f"DataLoaders prêts : lots étiquetés={len(labeled_loader)}, non étiquetés={len(unlabeled_loader)}")

print("Démarrage de la boucle d’entraînement")

# FixMatch training as a function (to unify with other methods)

def train_fixmatch(model, labeled_loader, unlabeled_loader, epochs=30, threshold=0.95, unsupervised_weight=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
    sup_crit = nn.CrossEntropyLoss()
    unsup_crit = nn.CrossEntropyLoss(reduction='none')
    for epoch in tqdm(range(epochs), desc='Entraînement FixMatch'):
        model.train()
        batch_iterator = zip(labeled_loader, unlabeled_loader)
        for (labeled_imgs, labels), (weak_unlabeled, strong_unlabeled) in batch_iterator:
            optimizer.zero_grad()
            # Supervised loss
            logits_sup = model(labeled_imgs)
            loss_sup = sup_crit(logits_sup, labels.squeeze())
            # Pseudo-labels from weak
            with torch.no_grad():
                logits_weak = model(weak_unlabeled)
                probs = F.softmax(logits_weak, dim=1)
                max_probs, pseudo_labels = torch.max(probs, dim=1)
                mask = max_probs.ge(threshold).float()
            # Unsupervised on strong
            logits_strong = model(strong_unlabeled)
            loss_unsup_raw = unsup_crit(logits_strong, pseudo_labels)
            loss_unsup = (loss_unsup_raw * mask).mean()
            # Total
            total_loss = loss_sup + unsupervised_weight * loss_unsup
            total_loss.backward()
            optimizer.step()
    return model

Taille du dataset étiqueté : 500
Taille du dataset non étiqueté : 6507
DataLoaders prêts : lots étiquetés=32, non étiquetés=102
Démarrage de la boucle d’entraînement


In [7]:
from cv2 import threshold
# fixation des paramétres d'entrainement
EPOCHS = 50
THRESHOLD= 0.95
UNSUPERVISED_WEIGHT = 1.0

print("Début de l’entraînement FixMatch…")
fix_model = SimpleCNN(in_channels=3, num_classes=n_classes)
fix_model = train_fixmatch(fix_model, labeled_loader, unlabeled_loader, epochs=EPOCHS, threshold=THRESHOLD, unsupervised_weight=UNSUPERVISED_WEIGHT)

Début de l’entraînement FixMatch…


Entraînement FixMatch: 100%|██████████| 50/50 [01:35<00:00,  1.91s/it]


## ⚙️ 2.2 Boucle d’entraînement FlexMatch
FlexMatch adapte le seuil dynamiquement par classe pour gérer les jeux de données déséquilibrés.

Instructions :

- Calculer la perte supervisée comme précédemment.
- Générer des pseudo‑labels avec un seuil dynamique : utiliser la probabilité maximale moyenne par classe comme seuil.
- Calculer la perte non supervisée avec le masque dynamique.
Combiner et rétropropager.

In [8]:
def train_flexmatch(model, labeled_loader, unlabeled_loader, epochs=20, threshold=0.95, unsupervised_weight=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
    sup_crit = nn.CrossEntropyLoss()
    unsup_crit = nn.CrossEntropyLoss(reduction='none')
    ema_conf = torch.full((n_classes,), 0.7)
    ema_m = 0.9
    for epoch in tqdm(range(epochs), desc='Entraînement FlexMatch'):
        model.train()
        for (labeled_imgs, labels), (weak_unlabeled, strong_unlabeled) in zip(labeled_loader, unlabeled_loader):
            optimizer.zero_grad()
            # Supervised
            logits_sup = model(labeled_imgs)
            loss_sup = sup_crit(logits_sup, labels.squeeze())
            # Weak preds
            with torch.no_grad():
                logits_weak = model(weak_unlabeled)
                probs = F.softmax(logits_weak, dim=1)
                max_probs, pseudo_labels = torch.max(probs, dim=1)
                # Update class-wise EMA confidence using samples of each predicted class
                for k in range(n_classes):
                    mask_k = (pseudo_labels == k)
                    if mask_k.any():
                        conf_k = max_probs[mask_k].mean()
                        ema_conf[k] = ema_m * ema_conf[k] + (1 - ema_m) * conf_k
                # Class-wise dynamic thresholds
                max_ema = torch.clamp(ema_conf.max(), min=1e-6)
                tau_k = threshold * (max_ema / torch.clamp(ema_conf, min=1e-6))
                eff_thresh = tau_k[pseudo_labels]
                mask = max_probs.ge(eff_thresh).float()
            # Unsupervised loss on strong views
            logits_strong = model(strong_unlabeled)
            loss_unsup_raw = unsup_crit(logits_strong, pseudo_labels)
            loss_unsup = (loss_unsup_raw * mask).mean()
            # Total
            total_loss = loss_sup + unsupervised_weight * loss_unsup
            total_loss.backward()
            optimizer.step()
    return model

In [9]:
# Train and evaluate FlexMatch
print("Début de l’entraînement FlexMatch…")
flex_model = SimpleCNN(in_channels=3, num_classes=n_classes)
flex_model = train_flexmatch(flex_model, labeled_loader, unlabeled_loader, epochs=EPOCHS, threshold=THRESHOLD, unsupervised_weight=UNSUPERVISED_WEIGHT)

Début de l’entraînement FlexMatch…


Entraînement FlexMatch: 100%|██████████| 50/50 [01:33<00:00,  1.87s/it]


## ⚙️ 2.3 Boucle d’entraînement MixMatch
MixMatch combine données étiquetées et non étiquetées via MixUp et un « sharpening » des probabilités.

Instructions :

- Calculer la perte supervisée sur les données étiquetées.
- Générer des pseudo‑labels avec sharpening (adoucir/affiner les probabilités avec une température).
- Mélanger données étiquetées et non étiquetées avec MixUp.
- Calculer la perte non supervisée sur les données mélangées.
- Combiner et rétropropager.

In [10]:
def one_hot(labels, num_classes):
    y = torch.zeros(labels.size(0), num_classes, device=labels.device)
    return y.scatter_(1, labels.view(-1, 1).long(), 1)

def sharpen(p, T=0.5):
    p_power = p ** (1.0 / T)
    return p_power / p_power.sum(dim=1, keepdim=True)

def soft_cross_entropy(logits, soft_targets):
    log_probs = F.log_softmax(logits, dim=1)
    return -(soft_targets * log_probs).sum(dim=1)

def train_mixmatch(model, labeled_loader, unlabeled_loader, epochs=200, alpha=0.75, T=0.5, lambda_u=100.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
    for epoch in tqdm(range(epochs), desc='Entraînement MixMatch'):
        model.train()
        for (labeled_imgs, labels), (u_imgs_w, _) in zip(labeled_loader, unlabeled_loader):
            b_l = labeled_imgs.size(0)
            b_u = u_imgs_w.size(0)
            # Guess labels for unlabeled
            with torch.no_grad():
                logits_u = model(u_imgs_w)
                probs_u = F.softmax(logits_u, dim=1)
                q_u = sharpen(probs_u, T)
            # One-hot for labeled
            y_l = one_hot(labels.squeeze(), n_classes)
            # Concatenate
            X = torch.cat([labeled_imgs, u_imgs_w], dim=0)
            Y = torch.cat([y_l, q_u], dim=0)
            # MixUp
            idx = torch.randperm(X.size(0))
            lam = np.random.beta(alpha, alpha)
            lam = max(lam, 1 - lam)
            X_mixed = lam * X + (1 - lam) * X[idx]
            Y_mixed = lam * Y + (1 - lam) * Y[idx]
            # Forward
            logits = model(X_mixed)
            # Losses
            loss_sup = soft_cross_entropy(logits[:b_l], Y_mixed[:b_l]).mean()
            probs_mixed = F.softmax(logits[b_l:], dim=1)
            loss_unsup = F.mse_loss(probs_mixed, Y_mixed[b_l:])
            loss = loss_sup + lambda_u * loss_unsup
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model

In [11]:
# Train and evaluate MixMatch
print("Début de l’entraînement MixMatch…")
mix_model = SimpleCNN(in_channels=3, num_classes=n_classes)
mix_model = train_mixmatch(mix_model, labeled_loader, unlabeled_loader, epochs=EPOCHS, alpha=0.75, T=0.5, lambda_u=50.0)

Début de l’entraînement MixMatch…


Entraînement MixMatch: 100%|██████████| 50/50 [01:26<00:00,  1.73s/it]


## 3. Évaluation finale et rétrospective
Évaluons tous les modèles et comparons leurs performances.

In [12]:
@torch.no_grad()
def evaluate_model(model, test_dataset, data_flag):
    model.eval()
    y_true = torch.tensor([])
    y_score_logits = torch.tensor([])
    y_score_preds = torch.tensor([])
    test_loader = DataLoader(test_dataset, batch_size=128)
    for images, labels in test_loader:
        outputs = model(images)
        y_true = torch.cat((y_true, labels), 0)
        y_score_logits = torch.cat((y_score_logits, outputs), 0)
        preds = torch.argmax(outputs, dim=1)
        y_score_preds = torch.cat((y_score_preds, preds), 0)
    y_true_np = y_true.squeeze().cpu().numpy()
    y_score_logits_np = y_score_logits.detach().cpu().numpy()
    y_score_preds_np = y_score_preds.detach().cpu().numpy()
    evaluator = Evaluator(data_flag, 'test')
    metrics = evaluator.evaluate(y_score_logits_np)
    f1_macro = f1_score(y_true_np, y_score_preds_np, average='macro')
    f1_weighted = f1_score(y_true_np, y_score_preds_np, average='weighted')
    return metrics[0], metrics[1], f1_macro, f1_weighted

In [13]:
# Consolidated Evaluation
print("Début de l’évaluation consolidée pour FixMatch, FlexMatch et MixMatch…")
results = []
for name, mdl in [("FixMatch", fix_model), ("FlexMatch", flex_model), ("MixMatch", mix_model)]:
    auc, acc, f1_macro, f1_weighted = evaluate_model(mdl, test_dataset, data_flag)
    results.append((name, auc, acc, f1_macro, f1_weighted))
    print(f"--- Résultats {name} ---")
    print(f"AUC : {auc:.3f}, Accuracy : {acc:.3f}, F1 (macro) : {f1_macro:.3f}, F1 (pondéré) : {f1_weighted:.3f}")

Début de l’évaluation consolidée pour FixMatch, FlexMatch et MixMatch…
--- Résultats FixMatch ---
AUC : 0.826, Accuracy : 0.671, F1 (macro) : 0.347, F1 (pondéré) : 0.658
--- Résultats FlexMatch ---
AUC : 0.811, Accuracy : 0.681, F1 (macro) : 0.355, F1 (pondéré) : 0.657
--- Résultats MixMatch ---
AUC : 0.788, Accuracy : 0.671, F1 (macro) : 0.143, F1 (pondéré) : 0.540


Vue d'ensemble et Performance Générale :

Les trois approches de semi-supervisé (SSL) ont été testées. FixMatch et FlexMatch montrent des performances comparables et généralement meilleures que MixMatch dans cette configuration. L'Accuracy (précision globale) pour FixMatch (0.671), FlexMatch (0.681) et MixMatch (0.671) est assez similaire, ce qui pourrait suggérer que les modèles apprennent à classer correctement la majorité des échantillons.

Analyse des Métriques Clés :

AUC (Area Under the Curve) :

FixMatch (0.826) et FlexMatch (0.811) ont des scores AUC respectables, indiquant une bonne capacité des modèles à distinguer les classes, même avec des données déséquilibrées. Un AUC supérieur à 0.8 est généralement considéré comme bon.
MixMatch (0.788) a un AUC légèrement inférieur, ce qui suggère une capacité de discrimination légèrement moins bonne comparée aux deux autres.
F1-Score (Macro vs. Pondéré) : C'est ici que réside l'information la plus critique, surtout pour un dataset comme DermaMNIST qui est souvent déséquilibré.

Le F1-pondéré (weighted F1-score) pour FixMatch (0.658) et FlexMatch (0.657) est proche de l'accuracy, ce qui est attendu car il prend en compte la proportion de chaque classe. Cela signifie que les modèles se débrouillent bien sur les classes majoritaires.
Cependant, le F1-macro (macro F1-score) est beaucoup plus révélateur de la performance sur les classes minoritaires, car il calcule le F1-score pour chaque classe et en fait la moyenne simple, traitant ainsi toutes les classes de manière égale.
FixMatch (0.347) et FlexMatch (0.355) ont des F1-macro relativement faibles. Cela indique clairement que même s'ils gèrent bien les classes majoritaires, ils ont des difficultés significatives avec les classes minoritaires. FlexMatch, avec son seuillage dynamique, semble un peu mieux s'en sortir sur cet aspect.
MixMatch (0.143) affiche un F1-macro extrêmement faible. C'est une alerte majeure : malgré une accuracy globale similaire, MixMatch semble échouer presque complètement à identifier les échantillons des classes minoritaires. Cela rend ce modèle pratiquement inutilisable pour des applications où toutes les classes sont importantes, même si elles sont rares.
Conclusion et Recommandations (Point de vue expert) :

FixMatch et FlexMatch sont les meilleurs performeurs dans cette expérience. FlexMatch montre un léger avantage en F1-macro, ce qui pourrait être dû à sa gestion dynamique des seuils, potentiellement plus robuste aux déséquilibres de classes. Cependant, les F1-macro scores restent globalement bas pour les deux, soulignant un problème persistant avec les classes minoritaires. Cela est courant dans les problèmes de classification médicale où certaines maladies (classes) sont rares.
MixMatch sous-performe de manière significative, en particulier sur les classes minoritaires comme l'indique son très faible F1-macro. Il serait intéressant de comprendre pourquoi, cela pourrait être lié à l'alpha (paramètre Beta distribution pour MixUp), la température de sharpening, ou le poids de la perte non supervisée (lambda_u), qui pourraient ne pas être optimaux pour ce dataset.